# Feature Engineering

## Team Selection

Choose the team, Grand Prix, and season that have already been processed. These parameters define the exact comparison window for the analysis.

In [1]:
team = "Aston Martin"
grand_prix = "Japanese Grand Prix"
year = 2025

## Imports and Configuration

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [2]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Read the cached processed laps for both teammates so the notebook can work from already cleaned telemetry.

In [3]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, team)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(year, grand_prix, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {year} {grand_prix} {driver_1} and {driver_2} not found: {e}")

req         WARNING 	DEFAULT CACHE ENABLED! (598.78 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']



Processed data for 2025 Japanese Grand Prix ALO and STR found in cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1


## Compute Corner Features

In [4]:
try:
    corners_1 = analyze_corners(lap_1["Telemetry"], turns)
    corners_2 = analyze_corners(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Corners analysis failed: {e}")
    
print(f"Corners data for {driver_1}:\n\n{corners_1}\n")
print(f"Corners data for {driver_2}:\n\n{corners_2}")

Corners data for ALO:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  316.703833    226.000000  216.639986
1      2  216.639986    175.000000  189.173638
2      3  237.515098    239.352615  254.000000
3      4  249.564264    225.497194  230.219390
4      5  240.460761    228.000000  230.293099
5      6  239.000000    193.341006  192.376692
6      7  247.666981    248.441726  264.163814
7      8  287.578909    255.677180  253.292756
8      9  220.620398    131.076242  157.962637
9     10  259.656799    260.756940  242.368613
10    11  155.684291     71.959831  129.241177
11    12  232.636890    235.726757  251.000000
12    13  275.989707    227.644628  230.295695
13    14  205.816865    169.863700  197.859438
14    15  301.000000    295.144680  295.579633
15    16  113.141150     88.000000  128.861125
16    17  106.697320    112.183443  199.647626
17    18  224.014809    227.351315  258.261961

Corners data for STR:

    Turn  EntrySpeed  MinimumSpeed   ExitSpeed
0      1  310

## Compute Braking features

In [5]:
try:
    braking_zones_1 = analyze_braking_zones(lap_1["Telemetry"])
    braking_zones_2 = analyze_braking_zones(lap_2["Telemetry"])
except Exception as e:
    print(f"Braking zones analysis failed: {e}")

try:
    braking_zones_1 = assign_braking_zones_to_turns(braking_zones_1, turns)
    braking_zones_2 = assign_braking_zones_to_turns(braking_zones_2, turns)
except Exception as e:
    print(f"Assignment of braking zones to turns failed: {e}")
    
print(f"Braking zones for {driver_1}:\n\n: {braking_zones_1}\n")
print(f"Braking zones for {driver_2}:\n\n: {braking_zones_2}")

Braking zones for ALO:

:    Turn  StartDistance  EndDistance  BrakingDuration  BrakingDistance  \
0     1     679.970542   782.977661         1.512405       103.007119   
1     6    1469.691790  1512.611423         0.713592        42.919633   
2     8    2233.661258  2285.164818         0.719429        51.503560   
3     9    2345.252304  2413.923717         1.426009        68.671413   
4    11    2791.616487  2877.455754         2.424912        85.839266   
5    13    3718.680561  3787.351974         0.976355        68.671413   
6    14    3856.023387  3924.694800         1.241569        68.671413   
7    16    5255.203424  5341.042690         2.124844        85.839266   

   BrakingPoint  
0     38.894948  
1     51.817657  
2     32.181384  
3     67.416182  
4     98.052495  
5     66.154317  
6     91.309722  
7    124.411490  

Braking zones for STR:

:     Turn  StartDistance  EndDistance  BrakingDuration  BrakingDistance  \
0      1     688.554468   791.561588         1.547558

## Compute Acceleration features

In [6]:
try:
    acceleration_zones_1 = analyze_acceleration_zones(lap_1["Telemetry"], braking_zones_1)
    acceleration_zones_2 = analyze_acceleration_zones(lap_2["Telemetry"], braking_zones_2)
except Exception as e:
    print(f"Acceleration zones analysis failed: {e}")
    
try:
    acceleration_zones_1 = analyze_throttle_lifts(lap_1["Telemetry"], acceleration_zones_1)
    acceleration_zones_2 = analyze_throttle_lifts(lap_2["Telemetry"], acceleration_zones_2)
except Exception as e:
    print(f"Throttle lifts analysis failed: {e}")

print(f"Acceleration zones for {driver_1}:\n\n: {acceleration_zones_1}\n")
print(f"Acceleration zones for {driver_2}:\n\n: {acceleration_zones_2}\n")

Acceleration zones for ALO:

:    Turn  StartDistance  EndDistance  AccelerationDuration  \
0     1     851.649074  1461.107863              9.580394   
1     6    1572.698909  2225.077331              9.138620   
2     8    2336.668377  2336.668377              0.000000   
3     9    2431.091570  2783.032561              5.745749   
4    11    2894.623607  3710.096635             12.669809   
5    13    3813.103754  3847.439460              0.536059   
6    14    3950.446580  5246.619497             16.812494   
7    16    5358.210543  5761.655094              7.845785   

   AccelerationDistance  TimeToFullThrottle  DistanceToFullThrottle  \
0            609.458789            0.678957               34.335706   
1            652.378422            0.466285               25.751780   
2              0.000000            0.000000                0.000000   
3            351.940991            0.837551               34.335706   
4            815.473028            2.087805               60.087

## Compute gear features

In [7]:
try:
    gear_shifts_1 = analyze_gear_shifts(lap_1["Telemetry"])
    gear_shifts_2 = analyze_gear_shifts(lap_2["Telemetry"])
except Exception as e:
    print(f"Gear shifts analysis failed: {e}")

print(f"Gear shifts for {driver_1}:\n\n: {gear_shifts_1}\n")
print(f"Gear shifts for {driver_2}:\n\n: {gear_shifts_2}\n")

Gear shifts for ALO:

:        Distance Direction  GearFrom  GearTo  RPMbefore  RPMafter  RPMChange
0     79.095679        up         7       8      11272     10696       -576
1    731.474101      down         8       7      10192      9910       -282
2    757.225881      down         7       6      10610     10364       -246
3    782.977661      down         6       5      10417     10691        274
4    971.824047        up         5       6      10924     10689       -235
5   1521.195349      down         6       5      11179     11051       -128
6   1667.122102        up         5       6      10868     10912         44
7   1795.881001        up         6       7      10799     10539       -260
8   2388.171937      down         7       4      10032      9962        -70
9   2422.507643      down         4       3      10022      9764       -258
10  2474.011203        up         3       4      11001     11033         32
11  2516.930836        up         4       5      10907     10849

In [8]:
lap_1["Corners"] = corners_1
lap_1["BrakingZones"] = braking_zones_1 
lap_1["AccelerationZones"] = acceleration_zones_1
lap_1["GearShifts"] = gear_shifts_1


lap_2["Corners"] = corners_2
lap_2["BrakingZones"] = braking_zones_2
lap_2["AccelerationZones"] = acceleration_zones_2
lap_2["GearShifts"] = gear_shifts_2

try:
    save_cache(year, grand_prix, segment, lap_1, lap_2)
except Exception as e:
    print(f"Failed to save lap cache: {e}")

Saving payload to C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_ALO.pkl
Saved: C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_ALO.pkl
Saving payload to C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_STR.pkl
Saved: C:\Users\joseg\OneDrive\Universidade\Outros\Projetos\f1-teammate-analysis\data\2025_japanese_grand_prix_Q1_STR.pkl

Processed data for 2025 Japanese Grand Prix saved to cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1
